# Weather Station Discovery — Multi-Station Experiment

**Source:** `wsi_cleaned.temp_observed_hourly` + `wsi_cleaned.temp_forecast_hourly`  
**Purpose:** Discover available weather stations, assess coverage, and recommend a station list for the multi-station spatial weather experiment.

### Sections
1. **Station Inventory** — What stations exist, row counts, date ranges
2. **Temporal Coverage** — Which stations cover the model's training period
3. **Inter-Station Correlation** — Are zone stations redundant with PJM aggregate?
4. **Spatial Spread Analysis** — Days with high cross-station temperature variance
5. **Forecast Table Coverage** — Do zone stations have D+1 forecasts?
6. **Recommended Station List** — Final selection for the experiment

## 1. Setup & Station Inventory

In [ ]:
import sys
from pathlib import Path
_BACKEND = str(Path().resolve().parent.parent.parent.parent)
if _BACKEND not in sys.path:
    sys.path.insert(0, _BACKEND)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import src.like_day_forecast.settings
from src.like_day_forecast.utils.azure_postgresql import pull_from_db
from src.like_day_forecast import configs

In [ ]:
# Query distinct stations from observed table
schema = configs.WEATHER_SCHEMA

query_observed = f"""
SELECT station_name,
       COUNT(*) as n_rows,
       COUNT(DISTINCT date) as n_dates,
       MIN(date) as first_date,
       MAX(date) as last_date,
       COUNT(DISTINCT hour_ending) as n_distinct_hours
FROM {schema}.temp_observed_hourly
GROUP BY station_name
ORDER BY station_name
"""

df_obs_stations = pull_from_db(query_observed)
print(f"Observed table — {len(df_obs_stations)} stations found")
display(df_obs_stations)

In [ ]:
# Query distinct stations from forecast table
query_forecast = f"""
SELECT station_name,
       COUNT(*) as n_rows,
       COUNT(DISTINCT date) as n_dates,
       MIN(date) as first_date,
       MAX(date) as last_date
FROM {schema}.temp_forecast_hourly
GROUP BY station_name
ORDER BY station_name
"""

df_fct_stations = pull_from_db(query_forecast)
print(f"Forecast table — {len(df_fct_stations)} stations found")
display(df_fct_stations)

In [ ]:
# Combined view: which stations have both observed AND forecast data
obs_set = set(df_obs_stations["station_name"])
fct_set = set(df_fct_stations["station_name"])

print(f"Observed only:  {obs_set - fct_set}")
print(f"Forecast only:  {fct_set - obs_set}")
print(f"Both tables:    {obs_set & fct_set}")
print(f"\nStations with both observed + forecast data are required for the experiment.")

## 2. Temporal Coverage

Check which stations cover the model's training period (`EXTENDED_FEATURE_START` = 2023-01-01).

In [ ]:
# Pull all station names that have data in the model training window
candidate_stations = sorted(obs_set & fct_set)
if not candidate_stations:
    candidate_stations = sorted(obs_set)
    print("⚠️ No stations have both observed + forecast — using observed-only stations")

print(f"Candidate stations: {candidate_stations}")

# Coverage heatmap: monthly row counts per station
stations_csv = ", ".join([f"'{s}'" for s in candidate_stations])
query_coverage = f"""
SELECT station_name,
       DATE_TRUNC('month', date) as month,
       COUNT(*) as n_rows
FROM {schema}.temp_observed_hourly
WHERE station_name IN ({stations_csv})
  AND date >= '2020-01-01'
GROUP BY station_name, DATE_TRUNC('month', date)
ORDER BY station_name, month
"""

df_coverage = pull_from_db(query_coverage)
df_coverage["month"] = pd.to_datetime(df_coverage["month"])

# Pivot and plot coverage heatmap
coverage_pivot = df_coverage.pivot_table(
    index="station_name", columns="month", values="n_rows", fill_value=0
)

# Expected rows per month (~720 for 30-day month)
coverage_pct = (coverage_pivot / 744 * 100).clip(upper=100).round(1)

fig = px.imshow(
    coverage_pct,
    labels=dict(x="Month", y="Station", color="Coverage %"),
    title="Monthly Data Coverage by Station (2020+)",
    color_continuous_scale="RdYlGn",
    zmin=0, zmax=100,
    aspect="auto",
)
fig.update_layout(height=max(300, len(candidate_stations) * 30))
fig.show()

In [ ]:
# Flag stations with poor coverage in the training window
training_start = pd.Timestamp(configs.EXTENDED_FEATURE_START)
training_months = coverage_pct.columns[coverage_pct.columns >= training_start]

if len(training_months) > 0:
    training_coverage = coverage_pct[training_months].mean(axis=1).round(1)
    print(f"Mean coverage since {configs.EXTENDED_FEATURE_START}:")
    for station, pct in training_coverage.sort_values(ascending=False).items():
        flag = "✅" if pct >= 95 else "⚠️" if pct >= 80 else "❌"
        print(f"  {flag} {station}: {pct}%")
else:
    print("No training window months found in coverage data")

## 3. Inter-Station Correlation

How correlated are zone stations with the PJM aggregate? Highly correlated stations add noise, not signal.

In [ ]:
# Pull daily avg temp per station for correlation analysis
query_daily = f"""
SELECT station_name, date, AVG(temperature) as temp_daily_avg
FROM {schema}.temp_observed_hourly
WHERE station_name IN ({stations_csv})
  AND date >= '2023-01-01'
GROUP BY station_name, date
ORDER BY station_name, date
"""

df_daily = pull_from_db(query_daily)
df_daily["date"] = pd.to_datetime(df_daily["date"])
print(f"Daily data: {len(df_daily):,} rows for {df_daily['station_name'].nunique()} stations")

# Pivot to wide format: one column per station
df_wide = df_daily.pivot_table(index="date", columns="station_name", values="temp_daily_avg")
print(f"Wide shape: {df_wide.shape}")
display(df_wide.tail())

In [ ]:
# Correlation matrix
corr = df_wide.corr().round(3)

fig = px.imshow(
    corr, text_auto=".3f",
    color_continuous_scale="RdBu_r", zmin=0.8, zmax=1.0,
    title="Inter-Station Daily Temperature Correlation (2023+)",
    aspect="auto",
)
fig.update_layout(height=max(500, len(candidate_stations) * 50))
fig.show()

# Correlation with PJM aggregate
if "PJM" in corr.columns:
    print("\nCorrelation with PJM aggregate:")
    pjm_corr = corr["PJM"].sort_values(ascending=True)
    for station, r in pjm_corr.items():
        if station == "PJM":
            continue
        flag = "🔵 High" if r > 0.98 else "🟢 Moderate" if r > 0.95 else "🟡 Distinct"
        print(f"  {flag} {station}: r = {r:.4f}")

## 4. Spatial Spread Analysis

Compute cross-station temperature spread and identify days where spatial variation is highest — these are the days where multi-station features should help most.

In [ ]:
# Compute spatial statistics per day
zone_cols = [c for c in df_wide.columns if c != "PJM"]
if len(zone_cols) < 2:
    print("⚠️ Need at least 2 zone stations for spatial analysis")
else:
    df_spatial = pd.DataFrame(index=df_wide.index)
    df_spatial["pjm_avg"] = df_wide["PJM"] if "PJM" in df_wide.columns else df_wide[zone_cols].mean(axis=1)
    df_spatial["station_std"] = df_wide[zone_cols].std(axis=1)
    df_spatial["station_range"] = df_wide[zone_cols].max(axis=1) - df_wide[zone_cols].min(axis=1)
    df_spatial["max_spread_vs_agg"] = df_wide[zone_cols].sub(df_spatial["pjm_avg"], axis=0).abs().max(axis=1)

    print("Daily Spatial Temperature Spread Statistics:")
    display(df_spatial[["station_std", "station_range", "max_spread_vs_agg"]].describe().round(2))

In [ ]:
# Spatial spread time series
if len(zone_cols) >= 2:
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=["PJM Aggregate Temperature", "Cross-Station Temperature Std"])

    fig.add_trace(go.Scatter(x=df_spatial.index, y=df_spatial["pjm_avg"],
                              mode="lines", line=dict(width=0.5, color="steelblue"),
                              name="PJM Avg"), row=1, col=1)

    fig.add_trace(go.Scatter(x=df_spatial.index, y=df_spatial["station_std"],
                              mode="lines", line=dict(width=0.5, color="firebrick"),
                              name="Station Std"), row=2, col=1)

    # Highlight high-spread days (>2 std above mean)
    threshold = df_spatial["station_std"].mean() + 2 * df_spatial["station_std"].std()
    high_spread = df_spatial[df_spatial["station_std"] > threshold]
    fig.add_trace(go.Scatter(x=high_spread.index, y=high_spread["station_std"],
                              mode="markers", marker=dict(color="red", size=4),
                              name=f"High spread (>{threshold:.1f}°F)"), row=2, col=1)

    fig.update_layout(height=500, title_text="Spatial Temperature Variation Over Time")
    fig.update_yaxes(title_text="Temp (°F)", row=1, col=1)
    fig.update_yaxes(title_text="Std (°F)", row=2, col=1)
    fig.show()

    print(f"High-spread days (>2σ): {len(high_spread)} out of {len(df_spatial)} ({len(high_spread)/len(df_spatial)*100:.1f}%)")

In [ ]:
# Seasonal pattern of spatial spread
if len(zone_cols) >= 2:
    df_spatial["month"] = df_spatial.index.month
    month_names = {1:"Jan",2:"Feb",3:"Mar",4:"Apr",5:"May",6:"Jun",
                   7:"Jul",8:"Aug",9:"Sep",10:"Oct",11:"Nov",12:"Dec"}
    df_spatial["month_name"] = df_spatial["month"].map(month_names)

    fig = px.box(df_spatial, x="month_name", y="station_std",
                 category_orders={"month_name": list(month_names.values())},
                 title="Monthly Distribution of Cross-Station Temperature Std",
                 labels={"station_std": "Station Std (°F)", "month_name": "Month"})
    fig.show()

## 5. Forecast Table Coverage

Check if zone-level forecasts are available in `temp_forecast_hourly`. If not, spatial features can only be used for reference-date (historical) matching, not D+1 target features.

In [ ]:
# Check forecast coverage per station
if len(df_fct_stations) > 0:
    print("Forecast table station coverage:")
    for _, row in df_fct_stations.iterrows():
        has_obs = row["station_name"] in obs_set
        print(f"  {'✅' if has_obs else '⚠️'} {row['station_name']}: "
              f"{row['n_dates']} dates ({row['first_date']} to {row['last_date']})")

    fct_stations = set(df_fct_stations["station_name"])
    zone_in_fct = [s for s in zone_cols if s in fct_stations]
    print(f"\nZone stations with forecasts: {zone_in_fct}")
    print(f"Zone stations without forecasts: {[s for s in zone_cols if s not in fct_stations]}")

    if len(zone_in_fct) >= 2:
        print("\n✅ Sufficient forecast coverage for D+1 spatial features")
    else:
        print("\n⚠️ Limited forecast coverage — spatial features will only work for historical reference dates")
else:
    print("❌ No forecast data found")

## 6. Recommended Station List

In [ ]:
# Selection criteria:
# 1. Must have observed data covering training window (>= 95% coverage since 2023)
# 2. Prefer stations with forecast data (for D+1 target features)
# 3. Include PJM aggregate as anchor
# 4. Prefer stations with lower correlation to PJM (more independent signal)

print("=" * 60)
print("  RECOMMENDED STATION LIST")
print("=" * 60)

recommended = ["PJM"]  # Always include aggregate

if "PJM" in corr.columns and len(zone_cols) >= 2:
    # Rank zones by: good coverage + low correlation with PJM + has forecasts
    zone_scores = []
    for station in zone_cols:
        r_pjm = corr.loc[station, "PJM"] if station in corr.index else 1.0
        cov = training_coverage.get(station, 0) if 'training_coverage' in dir() else 0
        has_fct = station in fct_set
        # Lower correlation = more value, higher coverage = better
        score = (1 - r_pjm) * 100 + (cov / 100) * 50 + (50 if has_fct else 0)
        zone_scores.append((station, r_pjm, cov, has_fct, score))

    zone_scores.sort(key=lambda x: x[4], reverse=True)

    print(f"\n{'Station':<15} {'r(PJM)':>8} {'Coverage':>10} {'Forecast':>10} {'Score':>8}")
    print("-" * 55)
    for station, r_pjm, cov, has_fct, score in zone_scores:
        selected = cov >= 90  # auto-select if good coverage
        if selected:
            recommended.append(station)
        marker = "→" if selected else " "
        print(f"{marker} {station:<13} {r_pjm:>8.4f} {cov:>9.1f}% {'✅' if has_fct else '❌':>10} {score:>8.1f}")

print(f"\n✅ Recommended stations ({len(recommended)}): {recommended}")
print(f"\nFor configs.py:")
print(f'WEATHER_STATIONS: list[str] = {recommended}')